# 04 - Resume Failed Indexing

Resumes a Nominatim import that was interrupted during the indexing phase.
This is the notebook equivalent of `scripts/04_resume_failed_indexing.sh`.

Use this if notebook 03 failed partway through indexing (e.g. SSL disconnect,
timeout, cluster preemption) and you want to pick up where it left off
without re-importing all the OSM data.

## Configuration

In [ ]:
dbutils.widgets.text("pg_project_id", "nominatim-lakebase", "Lakebase Project ID")
dbutils.widgets.text("pg_branch_id", "production", "Lakebase Branch ID")
dbutils.widgets.text("pg_user", "justin.monaldo@databricks.com", "PG User")
dbutils.widgets.text("pg_database", "nominatim", "PG Database")
dbutils.widgets.text("pg_port", "5432", "PG Port")
dbutils.widgets.text("resume_threads", "4", "Resume indexing threads")
dbutils.widgets.text("token_refresh_seconds", "2700", "Token refresh interval (seconds)")

In [ ]:
%run ./_helpers

In [ ]:
import os
import time
import threading
import subprocess
from pathlib import Path
from datetime import datetime

pg_project_id = dbutils.widgets.get("pg_project_id")
pg_branch_id = dbutils.widgets.get("pg_branch_id")
pg_user = dbutils.widgets.get("pg_user")
pg_database = dbutils.widgets.get("pg_database")
pg_port = dbutils.widgets.get("pg_port")
resume_threads = int(dbutils.widgets.get("resume_threads"))
TOKEN_REFRESH_SECONDS = int(dbutils.widgets.get("token_refresh_seconds"))

PROJECT_DIR = Path("/tmp/nominatim_project")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIR}\n"
        "Run 03_build_nominatim_server first (even a failed run creates this)."
    )

print(f"Project dir: {PROJECT_DIR}")
print(f"Threads:     {resume_threads}")
print(f"Branch:      projects/{pg_project_id}/branches/{pg_branch_id}")

## Authenticate and Setup pgpass

In [ ]:
pg_env = get_pg_env(
    project_id=pg_project_id,
    branch_id=pg_branch_id,
    user=pg_user,
    database=pg_database,
    port=pg_port,
)
pg_host = pg_env["PGHOST"]

pgpass_path = write_pgpass_file(pg_env)

# Build shell env
shell_env = {**os.environ, **pg_env}
shell_env["PGPASSFILE"] = pgpass_path
shell_env.pop("PGPASSWORD", None)

# Set NOMINATIM_DATABASE_DSN for the nominatim CLI
nominatim_dsn = (
    f"pgsql:dbname={pg_database};host={pg_host};port={pg_port};"
    f"user={pg_user};sslmode=require"
)
shell_env["NOMINATIM_DATABASE_DSN"] = nominatim_dsn

print(f"Token generated, pgpass: {pgpass_path}")
print(f"Database: {pg_user}@{pg_host}:{pg_port}/{pg_database}")

## Background Token Refresher

In [ ]:
refresher_stop = threading.Event()


def token_refresher_loop():
    while not refresher_stop.wait(timeout=TOKEN_REFRESH_SECONDS):
        try:
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"{ts} [Background] Refreshing OAuth token...")
            refresh_pgpass(pgpass_path, pg_env)
            print(f"{ts} [Background] Token refreshed")
        except Exception as e:
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"{ts} [Background] WARNING: token refresh failed: {e}")


refresher_thread = threading.Thread(target=token_refresher_loop, daemon=True)
refresher_thread.start()
print(f"Background token refresher started (interval: {TOKEN_REFRESH_SECONDS}s)")

## Resume Indexing

In [ ]:
def log(msg):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"{ts} {msg}")


log(f"Resuming import indexing with {resume_threads} threads...")

start_ts = time.time()
result = subprocess.run(
    [
        "nominatim", "import",
        "--continue", "indexing",
        "--threads", str(resume_threads),
    ],
    cwd=str(PROJECT_DIR),
    env=shell_env,
)
elapsed = int(time.time() - start_ts)

log(f"Resume indexing finished in {elapsed}s with exit code {result.returncode}")

## Cleanup

In [ ]:
# Stop refresher
refresher_stop.set()

# Clean up pgpass
try:
    os.remove(pgpass_path)
except OSError:
    pass

if result.returncode != 0:
    raise RuntimeError(f"Resume indexing failed with exit code {result.returncode}")
else:
    print()
    print("=" * 60)
    print("Resume indexing completed successfully!")
    print("=" * 60)